# iNaturalist batch metadata extraction

Loop over all CSV files under `data/experiment/inaturalist` (or the bundled `inaturalist_100_species_1000_obs` fallback), run the full pipeline with the `croissant_inaturalist_standard`, and save JSON outputs under `outputs/inat_<model_slug>/`.

Set `LLM_MODEL` (and provider) in `.env` before running.

In [1]:
import json
import logging
import re
import sys
from pathlib import Path

BASE = Path('..').resolve()
sys.path.insert(0, str(BASE))

from dotenv import load_dotenv

from src.config import LLM_PROVIDER, get_model_name
from src.orchestrator import run_metadata_extraction
from src.standards import METADATA_STANDARDS

load_dotenv(BASE / '.env')

STANDARD_NAME = 'croissant_inaturalist_standard'
TOPOLOGY = 'single'

LLM_NAME = get_model_name()
LLM_SLUG = re.sub(r'[^\w.\-]+', '_', LLM_NAME.replace('/', '_'))
EXPERIMENT_RUN = f'inat_{LLM_SLUG}'

INATURALIST_DIR_CANDIDATES = [
    BASE / 'data/experiment/inaturalist',
    BASE / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
]


def resolve_inaturalist_dir() -> Path:
    for path in INATURALIST_DIR_CANDIDATES:
        if path.is_dir():
            return path
    searched = '\n  '.join(str(p) for p in INATURALIST_DIR_CANDIDATES)
    raise FileNotFoundError(f'iNaturalist data directory not found. Tried:\n  {searched}')


def collect_inaturalist_csv_files(data_dir: Path) -> list[Path]:
    return sorted(data_dir.glob('*.csv'))


INATURALIST_DIR = resolve_inaturalist_dir()
OUTPUT_DIR = BASE / 'outputs' / EXPERIMENT_RUN
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

metadata_standard = METADATA_STANDARDS[STANDARD_NAME]


def output_path_for(csv_path: Path) -> Path:
    return OUTPUT_DIR / f'{csv_path.stem}_inaturalist.json'


all_csv_files = collect_inaturalist_csv_files(INATURALIST_DIR)
if not all_csv_files:
    raise FileNotFoundError(f'No CSV files found in {INATURALIST_DIR}')

csv_files = [p for p in all_csv_files if not output_path_for(p).exists()]
skipped_count = len(all_csv_files) - len(csv_files)

print(f'LLM provider: {LLM_PROVIDER}')
print(f'LLM model: {LLM_NAME}')
print(f'Experiment run: {EXPERIMENT_RUN}')
print(f'Data directory: {INATURALIST_DIR}')
print(f'Found {len(all_csv_files)} CSV file(s)')
print(f'Skipping {skipped_count} already completed in {OUTPUT_DIR}')
print(f'Remaining to run: {len(csv_files)}')
print(f'Output directory: {OUTPUT_DIR}')

LLM provider: surf
LLM model: Sehyo/Qwen3.5-122B-A10B-NVFP4
Experiment run: inat_Sehyo_Qwen3.5-122B-A10B-NVFP4
Data directory: /home/com3dian/Github/metadata_agent/data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs
Found 100 CSV file(s)
Skipping 0 already completed in /home/com3dian/Github/metadata_agent/outputs/inat_Sehyo_Qwen3.5-122B-A10B-NVFP4
Remaining to run: 100
Output directory: /home/com3dian/Github/metadata_agent/outputs/inat_Sehyo_Qwen3.5-122B-A10B-NVFP4


In [2]:
SPATIAL_KEYS = ('min_lat', 'min_lon', 'max_lat', 'max_lon')
TEMPORAL_KEYS = ('from', 'to')


def normalize_inaturalist_metadata(metadata):
    """Keep only schema-shaped iNaturalist Croissant metadata."""
    if not isinstance(metadata, dict):
        return None
    if not {'spatialCoverage', 'temporalCoverage'}.issubset(metadata.keys()):
        return None

    spatial = metadata.get('spatialCoverage')
    temporal = metadata.get('temporalCoverage')

    if spatial is not None:
        if not isinstance(spatial, dict) or not set(SPATIAL_KEYS).issubset(spatial.keys()):
            return None
        spatial = {key: spatial[key] for key in SPATIAL_KEYS}

    if temporal is not None:
        if not isinstance(temporal, dict) or not set(TEMPORAL_KEYS).issubset(temporal.keys()):
            return None
        temporal = {key: temporal[key] for key in TEMPORAL_KEYS}

    if spatial is None and temporal is None:
        return None

    return {
        'spatialCoverage': spatial,
        'temporalCoverage': temporal,
    }


def metadata_from_step_results(metadata_result):
    """Fallback when synthesis leaves metadata_output empty."""
    for step in reversed(metadata_result.step_results or []):
        if step.player_role != 'croissant_inaturalist_metadata_generator':
            continue
        for player_result in step.individual_results or []:
            analysis = player_result.get('analysis')
            candidates = [analysis]
            if isinstance(analysis, str):
                candidates = [analysis.strip()]
                start = analysis.find('{')
                end = analysis.rfind('}')
                if start != -1 and end > start:
                    candidates.append(analysis[start:end + 1])
            for candidate in candidates:
                if isinstance(candidate, dict):
                    normalized = normalize_inaturalist_metadata(candidate)
                elif isinstance(candidate, str) and candidate:
                    try:
                        normalized = normalize_inaturalist_metadata(json.loads(candidate))
                    except json.JSONDecodeError:
                        normalized = None
                else:
                    normalized = None
                if normalized:
                    return normalized
    return None


def extract_metadata(metadata_result):
    if metadata_result is None:
        return None

    candidates = []
    if metadata_result.final_metadata:
        candidates.append(metadata_result.final_metadata)

    workspace = metadata_result.final_workspace or {}
    if workspace.get('metadata_output') is not None:
        candidates.append(workspace.get('metadata_output'))

    for candidate in candidates:
        normalized = normalize_inaturalist_metadata(candidate)
        if normalized:
            return normalized

    return metadata_from_step_results(metadata_result)


def is_empty_metadata(metadata):
    return normalize_inaturalist_metadata(metadata) is None


def run_pipeline(csv_path: Path, attempt: int = 1):
    context_name = f'inaturalist_{csv_path.stem}'
    print(f'\n{"=" * 60}')
    print(f'[{attempt}] Processing {csv_path.name} (context: {context_name})')

    return run_metadata_extraction(
        source=str(csv_path.resolve()),
        metadata_standard=metadata_standard,
        metadata_standard_name=STANDARD_NAME,
        name=context_name,
        topology_name=TOPOLOGY,
    )


results_summary = []
success_count = 0

for csv_path in csv_files:
    entry = {
        'csv': csv_path.name,
        'success': False,
        'output_file': None,
        'attempts': 0,
        'error': None,
    }

    metadata = None
    for attempt in (1, 2):
        entry['attempts'] = attempt
        try:
            result = run_pipeline(csv_path, attempt=attempt)
            metadata = extract_metadata(result)
        except Exception as exc:
            entry['error'] = str(exc)
            print(f'  Attempt {attempt} failed: {exc}')
            continue

        if not is_empty_metadata(metadata):
            break

        print(f'  Attempt {attempt} returned empty metadata' + (' — retrying' if attempt == 1 else ''))

    if is_empty_metadata(metadata):
        print(f'  FAILED: no metadata for {csv_path.name}')
        results_summary.append(entry)
        continue

    output_file = output_path_for(csv_path)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    entry['success'] = True
    entry['output_file'] = str(output_file)
    success_count += 1
    print(f'  Saved: {output_file}')
    results_summary.append(entry)

print(f'\n{"=" * 60}')
print(f'Batch complete: {success_count}/{len(csv_files)} successful this run')
print(f'Previously completed (skipped): {skipped_count}')
print(f'Total with outputs: {skipped_count + success_count}/{len(all_csv_files)}')
for item in results_summary:
    status = 'OK' if item['success'] else 'FAIL'
    detail = item['output_file'] or item.get('error') or 'empty metadata'
    print(f'  [{status}] {item["csv"]} ({item["attempts"]} attempt(s)) -> {detail}')


[1] Processing acianthus_fornicatus.csv (context: inaturalist_acianthus_fornicatus)


/home/com3dian/Github/metadata_agent/src/orchestrator/orchestrator.py:400: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=topology_name)


2026-07-06 17:21:50,913 - INFO - root - PlanExecutor initialized with topology: single
2026-07-06 17:21:50,914 - INFO - root -   Players per step: 1
2026-07-06 17:21:50,914 - INFO - root -   Debate rounds: 0
2026-07-06 17:21:50,914 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-07-06 17:21:50,914 - INFO - root - Orchestrator initialized with topology: single
2026-07-06 17:21:50,915 - INFO - root - ============================================================
2026-07-06 17:21:50,915 - INFO - root - STARTING ORCHESTRATION
2026-07-06 17:21:50,915 - INFO - root - Context: inaturalist_acianthus_fornicatus
2026-07-06 17:21:50,915 - INFO - root - Type: single_csv
2026-07-06 17:21:50,916 - INFO - root - Resources: ['acianthus_fornicatus']
2026-07-06 17:21:50,916 - INFO - root - Metadata standard: croissant_inaturalist_standard (structured output)
2026-07-06 17

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-06 17:21:56,458 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-06 17:21:56,477 - WARNING - root - Plan validation warning: Step 4 ('generate_metadata') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-07-06 17:21:56,478 - INFO - root - Plan generated successfully!
2026-07-06 17:21:56,478 - INFO - root - Number of steps: 5
2026-07-06 17:21:56,478 - INFO - root -   Step 1: detect_spatial_columns (player: spatial_temporal_specialist)
2026-07-06 17:21:56,478 - INFO - root -   Step 2: detect_temporal_columns (player: spatial_temporal_specialist)
2026-07-06 17:21:56,478 - INFO - root -   Step 3: get_spatial_extent (player: spatial_temporal_specialist)
2026-07-06 17:21:56,478 - INFO - root -   Step 4: get_temporal_extent (player: spatial_temporal_specialist)
2026-07-06 17:21:56,478 - INFO - root -   Step 5: generate_metadata (player: croissant_inaturalist_metadata_generator)
2026-07-06 17:21:56,479 - IN

...

/home/com3dian/Github/metadata_agent/src/orchestrator/orchestrator.py:400: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=topology_name)


2026-07-06 17:29:25,166 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-06 17:29:25,179 - WARNING - root - Plan validation warning: Step 4 ('generate_metadata') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-07-06 17:29:25,180 - INFO - root - Plan generated successfully!
2026-07-06 17:29:25,180 - INFO - root - Number of steps: 5
2026-07-06 17:29:25,180 - INFO - root -   Step 1: detect_spatial_columns (player: spatial_temporal_specialist)
2026-07-06 17:29:25,180 - INFO - root -   Step 2: detect_temporal_columns (player: spatial_temporal_specialist)
2026-07-06 17:29:25,181 - INFO - root -   Step 3: get_spatial_extent (player: spatial_temporal_specialist)
2026-07-06 17:29:25,181 - INFO - root -   Step 4: get_temporal_extent (player: spatial_temporal_specialist)
2026-07-06 17:29:25,181 - INFO - root -   Step 5: generate_metadata (player: croissant_inaturalist_metadata_generator)
2026-07-06 17:29:25,181 - IN

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-06 17:29:26,575 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-06 17:29:26,577 - INFO - root - Player 'spatial_temporal_specialist_1': blocking finish — still need: detect_temporal_columns, detect_spatial_columns
2026-07-06 17:29:28,171 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-06 17:29:28,207 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 2 — model requested 1 tool(s): detect_temporal_columns
2026-07-06 17:29:29,275 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-06 17:29:29,307 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 3 — model requested 1 tool(s): detect_spatial_columns


KeyboardInterrupt: 